# TCC UTP — Treino YOLO no Google Colab

Detecção da classe única **fracture** no GRAZPEDWRI-DX com Ultralytics YOLO.

## Antes de abrir este notebook

1. **Runtime → Change runtime type → GPU** (T4 ou melhor).
2. Coloque o projeto no Google Drive (veja célula abaixo).

### Opção A — projeto completo no Drive (mais simples)

Copie a pasta `tcc__Copy` para o Drive, por exemplo:

`Meu Drive/TCC/tcc__Copy/` contendo `archive/`, `codigo/`, etc.

### Opção B — só o dataset `archive` no Drive

Se o projeto é grande, envie só `archive/` (zip) e clone/copie só `codigo/`.

### Opção C — datasets já preparados (mais rápido)

Se a preparação já foi executada no PC, copie `codigo/dataset/yolo/fracture_only/`
e `codigo/dataset/yolo_8bit/fracture_only/` para o Drive e pule a etapa 3.

In [ ]:
# Ajuste este caminho ao local do seu projeto no Google Drive
PROJECT_ROOT = "/content/drive/MyDrive/TCC/tcc__Copy"
CODIGO_DIR = f"{PROJECT_ROOT}/codigo"
ARCHIVE_DIR = f"{PROJECT_ROOT}/archive"

## 1. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
from pathlib import Path

for p in [PROJECT_ROOT, CODIGO_DIR, ARCHIVE_DIR]:
    print(p, "OK" if Path(p).exists() else "FALTANDO")

os.chdir(CODIGO_DIR)
print("Working dir:", os.getcwd())

## 2. Instalar dependências (GPU)

In [ ]:
!pip install -q --upgrade pip
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu124
!pip install -q ultralytics opencv-python pandas scikit-learn pyyaml tqdm matplotlib seaborn albumentations

In [ ]:
import torch
from ultralytics import YOLO

print("PyTorch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Ultralytics OK")

## 3. Preparar dataset YOLO (16-bit → 8-bit + CLAHE)

**Pule esta seção** se `dataset/yolo_8bit/` já existe (Opção C).

Demora ~1–2 h no dataset completo. No Colab use `--copy-images` (hardlink não funciona bem no Drive).

In [ ]:
# Gera os dois datasets comparáveis, ambos com uma única classe: fracture
PREPARE_FULL = False  # mude para True para processar as 20.327 imagens

limit = "" if PREPARE_FULL else " --max-images 20"
cmd_baseline = (
    "python -m src.prepare_dataset --classes fracture --copy-images "
    f"--archive {ARCHIVE_DIR}{limit}"
)
cmd_proposed = (
    "python -m src.prepare_dataset --classes fracture --preprocess --copy-images "
    f"--archive {ARCHIVE_DIR}{limit}"
)

print(cmd_baseline)
!{cmd_baseline}
print(cmd_proposed)
!{cmd_proposed}

In [ ]:
from pathlib import Path

for dataset in (
    Path("dataset/yolo/fracture_only"),
    Path("dataset/yolo_8bit/fracture_only"),
):
    print(f"\n{dataset}")
    for split in ("train", "val", "test"):
        n = len(list((dataset / "images" / split).glob("*.png")))
        print(f"  {split}: {n} imagens")

## 4. Treinar YOLO

Ajuste `EPOCHS` e `BATCH` se a GPU der erro de memória (tente `batch=8`).

In [ ]:
# Execute primeiro proposed; depois troque para baseline para a comparação do TCC
EXPERIMENT = "proposed"  # "proposed" ou "baseline"
EPOCHS = 100
BATCH = 16  # use 8 se OOM (out of memory)
RUN_NAME = EXPERIMENT

!python -m src.run_experiments --only {EXPERIMENT} --epochs {EPOCHS} --batch {BATCH} --device 0

## 5. Avaliar no conjunto de teste

In [ ]:
# run_experiments já avalia no conjunto test e salva o JSON de métricas.
WEIGHTS = f"runs/detect/{RUN_NAME}/weights/best.pt"
METRICS = f"runs/experiments/{RUN_NAME}/metrics_test.json"

import json
from pathlib import Path

print("Pesos:", WEIGHTS)
if Path(METRICS).exists():
    print(json.dumps(json.loads(Path(METRICS).read_text()), indent=2, ensure_ascii=False))

# Depois de executar baseline e proposed, gera a tabela usada no PDF:
!python -m src.export_metrics

## 6. Salvar pesos no Google Drive

Evita perder o modelo se a sessão Colab expirar.

In [ ]:
import shutil
from pathlib import Path

dest = Path(PROJECT_ROOT) / "colab_outputs" / RUN_NAME
dest.mkdir(parents=True, exist_ok=True)

src_run = Path(f"runs/detect/{RUN_NAME}")
for name in ("weights/best.pt", "weights/last.pt", "results.png", "confusion_matrix.png"):
    p = src_run / name
    if p.exists():
        shutil.copy2(p, dest / Path(name).name)
        print("Copiado:", p)

print("Salvo em:", dest)

## 7. Inferência de exemplo

In [ ]:
SAMPLE = list(Path("dataset/yolo_8bit/images/val").glob("*.png"))[0]
!python -m src.predict --weights {WEIGHTS} --source {SAMPLE} --preprocess --name colab_demo

In [ ]:
from IPython.display import Image, display
pred = list(Path("runs/predict/colab_demo").glob("*_pred.png"))
if pred:
    display(Image(filename=str(pred[0])))